## 표정 유사도 채점 v2 (웃음벨 - 표정 모듈)

기획서 4.2 정석: '생김새'가 아니라 '표정의 모양(blendshape 52계수)'을 비교.

**v2 변경점:** 전처리(얼굴 크롭 정규화) 셀을 분리 추가.
- 배경 제거는 *하지 않음* — face_landmarker는 어차피 얼굴만 잘라 분석하므로 배경은 무의미.
- 대신 목표/유저 얼굴을 같은 크기로 크롭·정규화 -> 패딩·배경·얼굴크기로 생기던 점수 흔들림 제거.

**채점:** 방향(cosine, 표정 종류) 0.4 + 모양·세기(가중오차) 0.6

**준비물:** `models/face_landmarker.task`, `data/origin.jpg`, `data/test_images/`
모델: https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


def imread_unicode(path):
    """cv2.imread는 윈도우 한글 경로에서 실패 -> np.fromfile + imdecode로 우회."""
    data = np.fromfile(path, dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def build_detector(model_path="models/face_landmarker.task"):
    """blendshape(표정 계수 52개) 출력을 켠 Face Landmarker 생성."""
    base = python.BaseOptions(model_asset_path=model_path)
    opts = vision.FaceLandmarkerOptions(
        base_options=base, output_face_blendshapes=True, num_faces=1
    )
    return vision.FaceLandmarker.create_from_options(opts)


def blendshapes_of(detector, bgr):
    """BGR 이미지 -> (52차원 표정계수 벡터[이름순], {이름:값}). 얼굴 없으면 (None, None)."""
    if bgr is None:
        return None, None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.face_blendshapes:
        return None, None
    shapes = sorted(res.face_blendshapes[0], key=lambda c: c.category_name)
    names = [c.category_name for c in shapes]
    vec = np.array([c.score for c in shapes], dtype=np.float32)
    return vec, dict(zip(names, vec))


def top_expressions(shape_dict, k=5):
    items = sorted(shape_dict.items(), key=lambda kv: kv[1], reverse=True)
    return [(n, round(float(v), 2)) for n, v in items[:k] if v > 0.05]


def score_expression(target, user, k=180.0, w_dir=0.4, w_shape=0.6):
    """방향(cosine)=표정 종류 + 모양·세기(가중오차)=같은 근육 같은 세기 -> 0~100."""
    cos = float(np.dot(target, user) / (np.linalg.norm(target) * np.linalg.norm(user) + 1e-8))
    cos_score = max(0.0, cos) * 100
    w = np.maximum(target, user)
    werr = float(np.sum(w * np.abs(target - user)) / (np.sum(w) + 1e-8))
    shape_score = max(0.0, 100 - werr * k)
    final = w_dir * cos_score + w_shape * shape_score
    return {"final": round(final, 1), "cosine": round(cos_score, 1),
            "shape": round(shape_score, 1), "werr": round(werr, 3)}

In [ ]:
# ===== 전처리: 얼굴 크롭 정규화 =====
# 배경을 지우는 게 아니라, 얼굴 '크기·위치'를 통일한다. 목표/유저에 똑같이 적용하면
# 패딩·배경·얼굴크기 때문에 생기던 세기 흔들림이 사라진다.

def detect_face_box(detector, bgr, margin=0.4):
    """얼굴 랜드마크로 얼굴 영역을 찾아 margin만큼 여백 준 정사각 박스(픽셀 좌표) 반환."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.face_landmarks:
        return None
    h, w = bgr.shape[:2]
    xs = np.array([lm.x for lm in res.face_landmarks[0]]) * w
    ys = np.array([lm.y for lm in res.face_landmarks[0]]) * h
    cx, cy = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
    side = max(xs.max() - xs.min(), ys.max() - ys.min()) * (1 + margin)
    half = side / 2
    return (int(round(cx - half)), int(round(cy - half)),
            int(round(cx + half)), int(round(cy + half)))


def preprocess_face(detector, bgr, out_size=256, margin=0.4):
    """얼굴을 정사각으로 잘라 out_size로 통일. 얼굴 못 찾으면 None."""
    box = detect_face_box(detector, bgr, margin)
    if box is None:
        return None
    x1, y1, x2, y2 = box
    # 박스가 프레임을 벗어나면 검은 여백으로 안전하게 채움
    pad = max(0, -x1, -y1, x2 - bgr.shape[1], y2 - bgr.shape[0])
    if pad > 0:
        bgr = cv2.copyMakeBorder(bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))
        x1 += pad; y1 += pad; x2 += pad; y2 += pad
    crop = bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return cv2.resize(crop, (out_size, out_size))


def expression_vector(detector, image_path, use_preprocess=True, margin=0.4):
    """경로 -> (선택)전처리 -> 표정계수 (벡터, 딕셔너리)."""
    bgr = imread_unicode(image_path)
    if bgr is None:
        return None, None
    if use_preprocess:
        crop = preprocess_face(detector, bgr, margin=margin)
        if crop is not None:
            bgr = crop  # 정규화된 얼굴로 교체 (얼굴 못 찾으면 원본 그대로)
    return blendshapes_of(detector, bgr)


def evaluate_expression_similarity(target_path, user_path, detector, k=180.0, use_preprocess=True):
    """팀원 evaluate_pose_similarity와 동일한 (점수, 메시지) 반환 -> 나중에 합산 쉬움."""
    tv, td = expression_vector(detector, target_path, use_preprocess)
    uv, ud = expression_vector(detector, user_path, use_preprocess)
    if tv is None:
        return 0.0, f"인식 실패(목표): {target_path}"
    if uv is None:
        return 0.0, f"인식 실패(유저): {user_path}"
    s = score_expression(tv, uv, k=k)
    msg = (f"성공 (방향 {s['cosine']} / 모양·세기 {s['shape']})  "
           f"목표:{top_expressions(td)}  유저:{top_expressions(ud)}")
    return s["final"], msg

In [ ]:
# 모델 로드 (한 번만 실행)
detector = build_detector("models/face_landmarker.task")
print("표정 채점기 준비 완료")

In [ ]:
import os, glob

# ── 튜닝 파라미터 (여기 바꿔가며 반복 실행) ──────────────
K = 180.0                        # 클수록 엄격(점수 차이 커짐)
USE_PREPROCESS = True            # 얼굴 크롭 정규화 on/off (전처리 효과 비교용)
target_img = "data/origin.jpg"   # 따라 할 목표 표정
test_dir   = "data/test_images"  # 채점할 유저 사진 폴더
# ───────────────────────────────────────────────

exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
files = sorted(p for p in glob.glob(os.path.join(test_dir, "*"))
               if p.lower().endswith(exts))

results = []
for p in files:
    sc, msg = evaluate_expression_similarity(target_img, p, detector, k=K, use_preprocess=USE_PREPROCESS)
    results.append((os.path.basename(p), sc, msg))
results.sort(key=lambda r: r[1], reverse=True)

print(f"목표: {target_img}  | 전처리(얼굴크롭): {USE_PREPROCESS}\n")
print(f"{'순위':<5}{'점수':>8}   파일명")
print("-" * 40)
for rank, (name, sc, msg) in enumerate(results, 1):
    print(f"{rank:<5}{sc:>8.2f}   {name}")
if results:
    print(f"\n🔥 1등: {results[0][0]} ({results[0][1]:.2f}점)")
    print(f"   상세: {results[0][2]}")